# Day 18 — Agricultural Commodity Price Analyzer

**Question:** how do maize, wheat and soybean prices move through the year and across years — seasonality, variability, and past shocks?

**Data:** IMF Primary Commodity Prices via FRED (no key), monthly 1992-01 → 2026-09, USD per metric ton. Weather: Open-Meteo ERA5 (Iowa + Illinois). Drought: US Drought Monitor state DSCI (Iowa + Illinois).

**Four limits, stated up front:** (1) prices are IMF *global benchmarks*, not USDA farm-gate prices (Quick Stats needs a key; none was available); (2) latest month is 2026-09 and 2026 is a partial year, excluded from annual statistics; (3) the weather/drought join is annual, 2-state, 26 years — associations, never causal claims; (4) the forecast is a statistical extrapolation, not a market prediction.

In [1]:
from pathlib import Path
import sys

def _find_root(marker="data/processed/monthly_prices.csv"):
    here = Path("__file__").resolve().parent if "__file__" in globals() else Path.cwd()
    for parent in [here, *here.parents]:
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(f"could not find {marker} above {here}")

ROOT = _find_root()
sys.path.insert(0, str(ROOT))
print("ROOT:", ROOT)

ROOT: /home/hatch/workspace/goals/60-day-data-ai-portfolio/day-18-agricultural-commodity-price-analyzer


## 1. The monthly price table — 417 months × 3 commodities

Log returns are ln(P_t / P_{t-1}) on monthly prices. The first month per commodity has no return.

In [2]:
import pandas as pd
monthly = pd.read_csv(ROOT / "data/processed/monthly_prices.csv", parse_dates=["date"])
print(monthly.shape)
monthly.head(6)

(1251, 4)


,date,commodity,price_usd_per_t,log_return
0,1992-01-01,maize,109.586357,NaN
1,1992-02-01,maize,113.921257,0.038795
2,1992-03-01,maize,116.684593,0.023967
3,1992-04-01,maize,109.055138,-0.067621
4,1992-05-01,maize,109.177910,0.001125
5,1992-06-01,maize,110.201927,0.009336


In [3]:
monthly.groupby("commodity")["price_usd_per_t"].agg(["min", "max", "mean"]).round(2)

,min,max,mean
commodity,,,
maize,75.06,348.51,166.14
soybeans,158.31,622.91,335.65
wheat,88.55,444.16,186.96


## 2. Price levels and past shocks

The 2007–08 food-price crisis, the 2012 drought, the 2020 COVID trough and the 2022 war spike are all visible in the levels. A shock month is |log return| ≥ 2× the rolling 24-month return standard deviation.

In [4]:
from IPython.display import Image
Image(str(ROOT / "outputs/charts/01_price_levels.png"))

In [5]:
from src import analysis as an
shocks = pd.read_csv(ROOT / "data/processed/shocks.csv", parse_dates=["date"])
print("shock months:", len(shocks))
shocks.sort_values("log_return", key=lambda s: s.abs(), ascending=False).head(10).round(4)

shock months: 76


/tmp/ipykernel_14169/4130147330.py:4: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  shocks.sort_values("log_return", key=lambda s: s.abs(), ascending=False).head(10).round(4)


,date,commodity,price_usd_per_t,log_return,rolling_24m_std,shock_flag,top10
63,2008-10-01,wheat,202.7749,-0.2602,0.1140,True,True
2,1996-09-01,maize,145.2392,-0.2518,0.0688,True,True
65,2012-07-01,wheat,310.7464,0.2502,0.0860,True,True
29,2004-08-01,soybeans,225.7900,-0.2495,0.0820,True,True
33,2008-10-01,soybeans,338.7849,-0.2463,0.0881,True,True
13,2008-10-01,maize,183.0247,-0.2455,0.0932,True,True
69,2017-08-01,wheat,139.0508,-0.2273,0.0746,True,True
14,2012-07-01,maize,332.9989,0.2197,0.0721,True,True
59,2007-09-01,wheat,297.9525,0.2134,0.0664,True,True
62,2008-04-01,wheat,326.8516,-0.2114,0.0914,True,True


In [6]:
an.named_window_changes(monthly)

{'2007-08 food-price crisis': {'start_price': 165.1600189849624,
  'end_price': 287.1320708928572,
  'pct_change': 73.85083427424419},
 '2012 US drought': {'start_price': 267.3289217687075,
  'end_price': 308.7504005639098,
  'pct_change': 15.494574444526465},
 '2020 COVID trough': {'start_price': 168.7445984962405,
  'end_price': 143.9892437500002,
  'pct_change': -14.670309430255235},
 '2022 Russia-Ukraine war spike': {'start_price': 292.67159304511273,
  'end_price': 335.7171340136054,
  'pct_change': 14.707796038769505}}

## 3. Seasonality — prices peak in late spring, trough at harvest

Seasonal index = mean price in that calendar month / overall mean price. The pattern holds on the last 10 full years too.

In [7]:
from IPython.display import Image
Image(str(ROOT / "outputs/charts/02_seasonality.png"))

In [8]:
seasonality = pd.read_csv(ROOT / "data/processed/seasonality.csv")
seasonality[seasonality["window"] == "full"].round(3)

,commodity,month,window,month_mean_price,seasonal_index,n_months
0,maize,1,full,165.603,0.997,35
1,maize,2,full,168.721,1.016,35
2,maize,3,full,170.537,1.026,35
3,maize,4,full,171.081,1.030,35
4,maize,5,full,172.293,1.037,35
5,maize,6,full,171.124,1.030,35
6,maize,7,full,166.835,1.004,35
7,maize,8,full,160.798,0.968,35
8,maize,9,full,160.792,0.968,35
9,maize,10,full,161.197,0.970,34


## 4. Volatility clusters around shocks

12-month rolling volatility (annualised %) peaks near 41% for maize in Nov 2008 and 47% for wheat in Oct 2008.

In [9]:
from IPython.display import Image
Image(str(ROOT / "outputs/charts/03_volatility_shocks.png"))

## 5. The weather/drought join — the honest null result

Across 26 annual points (2000–2025), YoY price change vs growing-season temperature, precipitation and DSCI: every |r| ≤ 0.15, every p-value above 0.46. The maize OLS regression on DSCI + precipitation anomaly has R² = 0.016. Global benchmark prices move on global supply and demand, not on two-state Corn Belt weather in the same year.

In [10]:
corr = pd.read_csv(ROOT / "data/processed/correlations.csv")
corr.round(4)

,commodity,driver,pearson_r,p_value,n
0,maize,gs_temp_mean_c,0.1430,0.4858,26
1,maize,gs_precip_total_mm,-0.0040,0.9845,26
2,maize,gs_dsci_mean,-0.0801,0.6974,26
3,wheat,gs_temp_mean_c,0.0814,0.6925,26
4,wheat,gs_precip_total_mm,-0.1502,0.4640,26
5,wheat,gs_dsci_mean,-0.0247,0.9045,26
6,soybeans,gs_temp_mean_c,-0.0457,0.8245,26
7,soybeans,gs_precip_total_mm,-0.0974,0.6360,26
8,soybeans,gs_dsci_mean,0.0066,0.9744,26


In [11]:
reg = pd.read_csv(ROOT / "data/processed/regression_maize.csv")
reg.round(4)

,intercept,coef_dsci,coef_precip_anomaly,r_squared,p_dsci,p_precip,n
0,9.8907,-0.0873,-0.0289,0.0156,0.552,0.6473,26


In [12]:
cross = pd.read_csv(ROOT / "data/processed/cross_correlations.csv")
cross.round(4)

,pair,window,pearson_r,p_value,n
0,maize-wheat,full,0.5457,0.0,416
1,maize-soybeans,full,0.6240,0.0,416
2,wheat-soybeans,full,0.4807,0.0,416
3,maize-wheat,last10y,0.5503,0.0,120
4,maize-soybeans,last10y,0.5607,0.0,120
5,wheat-soybeans,last10y,0.5091,0.0,120


## 6. Forecast — Holt-Winters on monthly maize prices

Backtest on 3 origins (fit through 2023-09, 2024-09, 2025-09; forecast the next 12 months each): Holt-Winters mean RMSE $23.38/t vs $41.74/t for the seasonal-naïve baseline. The 12-month forecast below is a statistical extrapolation, not a market prediction.

In [13]:
forecast = pd.read_csv(ROOT / "data/processed/forecast_maize.csv", parse_dates=["date"])
forecast.round(2)

/tmp/ipykernel_14169/1624187069.py:2: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  forecast.round(2)


,date,forecast,lo_80,hi_80
0,2026-10-01,239.56,206.42,272.69
1,2026-11-01,240.17,207.03,273.30
2,2026-12-01,240.87,207.74,274.01
3,2027-01-01,245.61,212.48,278.75
4,2027-02-01,248.74,215.61,281.87
5,2027-03-01,250.56,217.42,283.69
6,2027-04-01,251.10,217.96,284.23
7,2027-05-01,252.31,219.18,285.45
8,2027-06-01,251.14,218.01,284.28
9,2027-07-01,246.85,213.71,279.98


## 7. Counties — where the 2012 drought hit hardest

2012 is the worst drought year in the series (growing-season DSCI mean 200.4; precipitation 182.7 mm, −30.7% below the 2000–2025 mean). The three hardest-hit counties are all in southern Illinois.

In [14]:
county = pd.read_csv(ROOT / "data/processed/county_shock_2012.csv")
county.head(10).round(2)

,fips,county,state,gs_d2plus_pct_2012,rank
0,17127,Massac County,IL,73.91,1
1,17153,Pulaski County,IL,72.28,2
2,17069,Hardin County,IL,71.65,3
3,17151,Pope County,IL,71.28,4
4,17087,Johnson County,IL,68.47,5
5,17003,Alexander County,IL,68.03,6
6,17181,Union County,IL,63.72,7
7,17059,Gallatin County,IL,62.49,8
8,19025,Calhoun County,IA,61.53,9
9,19091,Humboldt County,IA,61.36,10


In [15]:
from IPython.display import Image
Image(str(ROOT / "outputs/charts/04_county_map.png"))

## 8. Conclusions

1. **Seasonality is small and pre-harvest:** maize peaks in May (index 1.037), troughs in September (0.968); soybeans swing most (amplitude 0.099), wheat least (0.048).
2. **The shocks dwarf the seasons:** worst drawdowns are wheat −69.7%, maize −63.2%, soybeans −51.0%; a single month moved prices up to 26%.
3. **No annual weather/drought pricing signal:** all 9 correlations are insignificant (p > 0.46) and the maize regression R² is 0.016. The 2012 drought shows in the *monthly* moves (maize +22% in July 2012), not in the annual join.
4. **Holt-Winters beats seasonal naïve** on the maize backtest (RMSE $23.38/t vs $41.74/t); the 12-month outlook is $239.56/t in Oct 2026 rising to $252.31/t in May 2027.

**Recommendation:** sell into the late-spring peak, keep storage for the harvest trough, and hedge the shock risk — do not price off an annual weather model.